In [4]:
import cv2
import tkinter as tk
from tkinter import filedialog, messagebox
from textblob import TextBlob
from PIL import Image, ImageTk

# Load Haar cascades for face + smile detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
smile_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_smile.xml")


# FUNCTION TO DETECT THE SENTIMENT OF A TEXT
def analyze_text():
    
    image_label.image = None       

    text = txtEnterText.get("1.0", tk.END).strip()
    if not text:
        messagebox.showwarning("Input Error", "Please enter some text!")
        return
    
    analysis = TextBlob(text)
    if analysis.sentiment.polarity > 0:
        sentiment = "Positive 😀"
    elif analysis.sentiment.polarity < 0:
        sentiment = "Negative 😞"
    else:
        sentiment = "Neutral 😐"
    
    result_label.config(text=f"Text Sentiment  : {sentiment}")

#FUNCTION TO DETECT SENTIMENT OF AN IMAGE
def analyze_image():

    txtEnterText.delete("1.0", tk.END)
    file_path = filedialog.askopenfilename(filetypes=[("Image Files", "*.jpg *.png *.jpeg")])
    if not file_path:
        return

    # Load image with OpenCV
    img = cv2.imread(file_path)
    if img is None:
        messagebox.showerror("File Error", "Unable to open the selected image.")
        return

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Detect faces (use a smaller scaleFactor for better accuracy; require a reasonable minSize)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80))

    # Default sentiment
    sentiment = "Neutral 😐"

    if len(faces) > 0:
        smile_found = False

        for (x, y, w, h) in faces:
            # Crop the face region
            face_gray = gray[y:y + h, x:x + w]

            # Focus on the lower half (mouth region) — this improves smile detection
            mouth_start_row = int(h * 0.55)
            mouth_gray = face_gray[mouth_start_row:h, :]

            # Skip if mouth region is empty
            if mouth_gray.size == 0:
                continue

            # Improve contrast
            try:
                mouth_gray = cv2.equalizeHist(mouth_gray)
            except Exception:
                pass

            # Width for the smile for Haar to detect smile
            target_w = 120
            if mouth_gray.shape[1] > 0 and mouth_gray.shape[1] != target_w:
                scale = target_w / float(mouth_gray.shape[1])
                new_h = max(24, int(mouth_gray.shape[0] * scale)) #Ensuring height isnt too small
                mouth_gray_resized = cv2.resize(mouth_gray, (target_w, new_h))
            else:
                mouth_gray_resized = mouth_gray

            # Detect smiles in the mouth 
            smiles = smile_cascade.detectMultiScale(
                mouth_gray_resized,
                scaleFactor=1.2,
                minNeighbors=7,
                minSize=(15, 15)
            )

            if len(smiles) > 0:
                smile_found = True
                break  # if any face has a smile, we call it positive

        if smile_found:
            sentiment = "Positive 😀"
        else:
            # faces found but no smile detected
            sentiment = "Negative 😞"

    else:
        # no face detected -> Neutral
        sentiment = "Neutral 😐"

    # Show Sentiment in GUI
    result_label.config(text=f"Image Sentiment: {sentiment}")

    # Show uploaded image in GUI
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb).resize((200, 200))

    # Resize image for the label while keeping aspect ratio
    pil_img.thumbnail((300, 300))
    img_tk = ImageTk.PhotoImage(pil_img)
    image_label.config(image=img_tk)
    image_label.image = img_tk



#USER INTERFACE
root = tk.Tk()
root.title("")
root.geometry("550x550")



#FRAME -- MAIN FRAME
main_frame = tk.Frame(root)
main_frame.grid(row=1,column=0,pady=20,sticky="n")


heading = tk.Frame(main_frame)
heading.grid(row=0,column=0,sticky="n",pady=10,padx=10)

lblEnterText = tk.Label(heading,text="IMAGE AND TEXT SENTIMENT ANALYSIS",font=("",14,"bold","underline"))
lblEnterText.grid(row=0,column=0,padx=10,sticky=tk.W)

#FRAME - TEXT ANALYSIS
textSentiment_Analysis = tk.LabelFrame(main_frame,text="Text Sentiment")
textSentiment_Analysis.grid(row=1,column=0,sticky="w",pady=10,padx=10)

#FRAME - IMAGE ANALYSIS
imageSentiment_Analysis = tk.LabelFrame(main_frame,text="Image Sentiment")
imageSentiment_Analysis.grid(row=2,column=0,sticky="w",pady=10,padx=10)


#FRAME - OUTPUT
output_frame = tk.LabelFrame(main_frame)
output_frame.grid(row=3,column=0,sticky="w",pady=10,padx=10)

#SECTION For Checking TEXT Sentiment
lblEnterText = tk.Label(textSentiment_Analysis,text="Enter Text : ")
lblEnterText.grid(row=0,column=0,pady=10,padx=10,sticky=tk.W)

txtEnterText = tk.Text(textSentiment_Analysis, height=2, width=40)
txtEnterText.grid(row=0,column=1,padx=5,pady=5,sticky=tk.W)

btnAddTask = tk.Button(textSentiment_Analysis,text="Get Sentiment",command=analyze_text)
btnAddTask.grid(row=0,column=2,sticky=tk.W)

#IMAGE
lblUploadImage = tk.Label(imageSentiment_Analysis,text="Upload Image : ")
lblUploadImage.grid(row=0,column=0,pady=10,padx=10,sticky=tk.W)

uploadImage_btn = tk.Button(imageSentiment_Analysis,text="Choose Image",command=analyze_image)
uploadImage_btn.grid(row=0,column=1,padx=(0,278))


# Result
lblResult = tk.Label(output_frame,text="RESULT",font=("Roboto Mono",14,"bold"))
lblResult.grid(row=0,column=0,pady=1,padx=(10,450),sticky=tk.W)

result_label = tk.Label(output_frame, text="Output will be displayed here!", font=("Roboto Mono",14,"bold"))
result_label.grid(row=1,column=0,pady=(5,30),padx=10,sticky=tk.W)

image_label = tk.Label(output_frame)
image_label.grid(row=2,column=0)


if __name__ == "__main__":

    root.mainloop()






In [2]:
pip install textblob

   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------- ----------------------- 262.1/625.0 kB ? eta -:--:--
   ---------------- ----------------------- 262.1/625.0 kB ? eta -:--:--
   ------------------------------- ------ 524.3/625.0 kB 498.3 kB/s eta 0:00:01
   ------------------------------- ------ 524.3/625.0 kB 498.3 kB/s eta 0:00:01
   ---------------------------------------- 625.0/625.0 kB 439.3 kB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.
